In [1]:
import sys
from pathlib import Path
import pandas as pd
import fitz  
import tiktoken
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))


from system.rag import (
    LLM,
    Approaches,
    RAGExperimentRunner,
    CSVProcessor,
    LangfairMetricsCalculator,
    LangfairRunner,
    ApproachRetrievers,
)

from system.preprocess import (
    PDFPreprocessConfig,
    PDFPreprocessor,
    CorpusBuilderConfig,
    CorpusBuilder,
)


from system.evaluation import (
    JudgeBatchConfig,
    JudgeBatchBuilder,
    BatchResultsConfig,
    BatchResultsExporter,
    JudgeMergeConfig,
    JudgeResultsMerger,
    CSVColumnMergeConfig, 
    CSVColumnMerger
)

# Check if Tokens will fit in the context window

## CROP PDFS

In [2]:
PDF_DIR = Path("../data/input/input_pdfs")

TARGETS = {
    "Mill (Bridgeport)": {
        "path": PDF_DIR / "Bridgeport Series 1 Milling manual with schematics.pdf",
        "crop_top": 0.04, "crop_bottom": 0.075, "crop_left": 0.0, "crop_right": 0.0,
    },
    "UR5e Cobot": {
        "path": PDF_DIR / "UR5e_Universal_Robots User Manual.pdf",
        "crop_percent": 0.075,
    },
}

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf_as_text(path) -> str:
    pages = PyMuPDFLoader(str(path)).load()
    return "\n\n".join(p.page_content for p in pages).strip()

mill_text = load_pdf_as_text(TARGETS["Mill (Bridgeport)"]["path"])
ur5e_text = load_pdf_as_text(TARGETS["UR5e Cobot"]["path"])

print(f"Mill  : {len(mill_text):,} chars")
print(f"UR5e  : {len(ur5e_text):,} chars")

Mill  : 123,442 chars
UR5e  : 276,138 chars


In [4]:
doc = fitz.open(TARGETS["UR5e Cobot"]["path"])

### Crop PDFs and extract text

In [5]:
OUTPUT_DIR = Path("../data/input/cropped_pdfs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CROPPED = {}
for label, cfg in TARGETS.items():
    out = OUTPUT_DIR / f"cropped_{cfg['path'].name}"
    PDFPreprocessor.crop_pdf(
        cfg["path"], out,
        crop_percent=cfg.get("crop_percent", 0.075),
        crop_top=cfg.get("crop_top"),
        crop_bottom=cfg.get("crop_bottom"),
        crop_left=cfg.get("crop_left"),
        crop_right=cfg.get("crop_right"),
    )
    CROPPED[label] = out
    print(f"Cropped {label} -> {out.name}")

Cropped Mill (Bridgeport) -> cropped_Bridgeport Series 1 Milling manual with schematics.pdf
Cropped UR5e Cobot -> cropped_UR5e_Universal_Robots User Manual.pdf


In [6]:
mill_cropped_text = load_pdf_as_text(CROPPED["Mill (Bridgeport)"])
ur5e_cropped_text = load_pdf_as_text(CROPPED["UR5e Cobot"])

print(f"Mill  (cropped): {len(mill_cropped_text):,} chars")
print(f"UR5e  (cropped): {len(ur5e_cropped_text):,} chars")

Mill  (cropped): 121,502 chars
UR5e  (cropped): 253,244 chars


In [7]:
enc = tiktoken.get_encoding("cl100k_base")

for label, cfg in TARGETS.items():
    orig = load_pdf_as_text(cfg["path"])
    cropped = load_pdf_as_text(CROPPED[label])
    orig_tokens = len(enc.encode(orig))
    crop_tokens = len(enc.encode(cropped))
    print(f"{label}")
    print(f"  Original : {orig_tokens:,} tokens")
    print(f"  Cropped  : {crop_tokens:,} tokens")
    print(f"  Diff     : {orig_tokens - crop_tokens:,} tokens removed")
    print()

Mill (Bridgeport)
  Original : 46,606 tokens
  Cropped  : 45,319 tokens
  Diff     : 1,287 tokens removed

UR5e Cobot
  Original : 68,416 tokens
  Cropped  : 60,754 tokens
  Diff     : 7,662 tokens removed



# Long-Context RAG Evaluation

In [8]:
print(ur5e_cropped_text)

riginal instructions (en)
PolyScope 5
User Manual
UR5e



The information contained herein is the property of Universal Robots A/S and shall not be reproduced in
whole or in part without prior written approval of Universal Robots A/S. The information herein is subject to
change without notice and should not be construed as a commitment by Universal Robots A/S. This
document is periodically reviewed and revised.
Universal Robots A/S assumes no responsibility for any errors or omissions in this document.
Copyright © 2009–2024 by Universal Robots A/S.
The Universal Robots logo is a registered trademark of Universal Robots A/S.



Contents
1. Liability and Intended Use
11
1.1. Limitation of Liability
11
1.2. Intended Use
11
2. Your Robot
13
2.1. Technical Specifications UR5e
17
2.2. Maximum Payload
18
2.3. Stopping Time and Stopping Distance
19
2.4. PolyScope Overview
26
2.4.1. Icons/Tabs On PolyScope
27
3. Safety
29
3.1. General
29
3.2. Safety Message Types
30
3.3. General Warnings and Ca

In [9]:
from system.utils import EnvironmentConfig, read_text

env_config = EnvironmentConfig()
rets = ApproachRetrievers(env_config)
rets.set_long_context_texts({
    "Mill (Bridgeport)": mill_cropped_text,
    "UR5e Cobot": ur5e_cropped_text,
})

## Run Experiment — Mill

In [11]:
runner = RAGExperimentRunner(
    retrievers=rets,
    num_replicates=1,
    approaches=Approaches.LONG_CONTEXT,
    models=LLM.GPT_5_MINI_2025_08_07 | LLM.GPT_5_NANO_2025_08_07,
    max_tokens_list=[5000],
    efforts=["low"],
    topk_list=[1],
    ans_instr_A=read_text("../data/prompts/ans_instr_A.txt"),
    fewshot_A=read_text("../data/prompts/fewshot_A.txt"),
    max_concurrent=5,
    max_chars_per_content=500_000,
    include_hits_text= False
)
print(runner)

RAGExperimentRunner Configuration
  Approaches       : ['long_context']
  Models           : ['gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07']
  Max tokens       : [5000]
  Efforts          : ['low']
  Top-k            : [1]
  Answer instr IDs : ['A']
  Few-shot IDs     : ['A']
  Replicates       : 1
  Max concurrent   : 5
  Max chars/content: 500,000
  Include hits text: False
  Min words subsplit: 3000


In [13]:
MILL_QA_test= Path("../data/QA/MILL/Mill Feedback Accepted_test.csv")
MILL_QA = Path("../data/QA/MILL/Mill Feedback Accepted.csv")
MILL_OUTPUT = Path("../data/results/RAG_Output/LongContext/MILL_LONG_CONTEXT_OUTPUT.csv")
await runner.run(MILL_QA_test, MILL_OUTPUT)

Total permutations: 2
Completed 1 runs for approach=long_context, model=gpt-5-mini-2025-08-07
Completed 2 runs for approach=long_context, model=gpt-5-nano-2025-08-07
All results written to ..\data\results\RAG_Output\LongContext\MILL_LONG_CONTEXT_OUTPUT.csv


[{'permutation_id': 'eyJtZXRhZGF0YSI6eyJhbnN3ZXJfaW5zdHJ1Y3Rpb25zX2lkIjoiQSIsImFwcHJvYWNoIjoibG9uZ19jb250ZXh0IiwiZWZmb3J0IjoibG93IiwiZmV3X3Nob3RfaWQiOiJBIiwibWF4X3Rva2VucyI6NTAwMCwibW9kZWwiOiJncHQtNS1uYW5vLTIwMjUtMDgtMDciLCJyZWFzb25pbmdfZWZmb3J0IjoibG93IiwidG9wX2siOjF9LCJydW5fdXVpZCI6IjgwMzA0OWExLTQxNjgtNGE5Mi04OWE3LWY5Y2NkZjllNmI1ZSJ9MkbeQ5CGThk',
  'time_started': '2026-03-06 09:40:47 EST',
  'time_ended': '2026-03-06 09:40:54 EST',
  'total_elapsed_time': '7.30 Seconds',
  'min_words_for_subsplit': 3000,
  'approach': 'long_context',
  'model': 'gpt-5-nano-2025-08-07',
  'max_tokens': 5000,
  'reasoning_effort': 'low',
  'top_k': 1,
  'answer_instructions_id': 'A',
  'few_shot_id': 'A',
  'replicate': 1,
  'question': 'What are the 5 main categories of safety labeling used in the manual of the bridgeport mill?',
  'gold_answer': 'DANGER\r\nDANGER indicates a hazardous situation that, if not avoided, will result in death or serious injury.\r\n\r\nWARNING\r\nWARNING indicates a hazard

## Run Experiment — UR5e

In [ ]:
UR5E_QA_raw = Path("../data/QA/COBOT/Final COBOT Lathe QA raw .csv")  # create this file with question,gold_answer columns
df= pd.read_csv(UR5E_QA_raw).head()
df_new = pd.DataFrame()
df_new["question"] = df["Lora Suggestions for Revised Question Wording"]
df_new["gold_answer "] = df["Revised Question Answer"]
print(df_new.isna().sum())
print(len(df_new) == len(df))

question        0
gold_answer     0
dtype: int64
True


In [ ]:
df_new.to_csv("../data/QA/COBOT/Final_COBOT_QA.csv", index=False)

In [ ]:
UR5E_QA = Path("../data/QA/COBOT/Final COBOT Lathe QA .csv")  # create this file with question,gold_answer columns
UR5E_OUTPUT = Path("../data/results/RAG_Output/LongContext/UR5E_LONG_CONTEXT_OUTPUT.csv")
await runner.run(UR5E_QA, UR5E_OUTPUT)

NameError: name 'runner' is not defined

# Compute Similarity Metrics

In [ ]:
metrics_runner = LangfairRunner(
    calculator=LangfairMetricsCalculator(),
    processor=CSVProcessor(),
    max_concurrent=500,
)
await metrics_runner.run(q_a_csv=MILL_OUTPUT, out_csv=None)
# await metrics_runner.run(q_a_csv=UR5E_OUTPUT, out_csv=None)

# Judge Batch (Helpfulness & Correctness)

In [ ]:
judge_config = JudgeBatchConfig(
    csv_path=MILL_OUTPUT,
    output_jsonl=Path("../data/results/batchprocess/MILL_LONG_CONTEXT_BATCH.jsonl"),
    judge_model="gpt-5",
    completion_window="24h",
    submit_to_openai=False,  # set True when ready
    env_file=None,
)
builder = JudgeBatchBuilder(judge_config)
result = builder.run()
print(f"Prepared {result['num_requests']} requests (submitted={result['submitted']})")

# Parse Judge Results
After batch completes, update `batch_id` below and uncomment.

In [ ]:
# csv_config = BatchResultsConfig(
#     batch_id="batch_xxx",  # replace with real batch ID
#     raw_jsonl_path=Path("../data/results/batchprocess/mill_lc_batch_raw.jsonl"),
#     json_output_path=Path("../data/results/batchprocess/mill_lc_batch.json"),
#     csv_output_path=Path("../data/results/batchprocess/mill_lc_batch.csv"),
# )
# BatchResultsExporter().download_records(csv_config)
#
# merge_config = JudgeMergeConfig(
#     input_csv=csv_config.csv_output_path,
#     output_csv=Path("../data/results/batchprocess/mill_lc_batch_wide.csv"),
# )
# JudgeResultsMerger().run(merge_config)
#
# final_merge = CSVColumnMergeConfig(
#     left_csv=Path(str(MILL_OUTPUT).replace(".csv", "_with_metrics.csv")),
#     right_csv=merge_config.output_csv,
#     output_csv=Path("../data/results/final_merged/mill_long_context.csv"),
#     on="permutation_id",
#     how="left",
#     exclude_columns=["q", "retrieved_files", "meta_hits_text"],
# )
# CSVColumnMerger().run(final_merge)

# Analysis

In [ ]:
from system.analysis import analyze_csv

analyze_csv(
    csv_input=MILL_OUTPUT,
    output_dir=Path("../data/results/final_merged/analysis_mill_long_context/"),
)
# analyze_csv(
#     csv_input=UR5E_OUTPUT,
#     output_dir=Path("../data/results/final_merged/analysis_ur5e_long_context/"),
# )